The goal of this notebook is to get the time delay between the (bugged timestamps in the) kinect stream and the other streams in the .xdf file. 

# The problem

Due to a bug in LSL_Kinect, the timestamps in the kinect streams are relative to the computer's startup time, whereas timestamps should be defined by LSL to enable synchronization with other streams.  
As a consequence, the kinect streams are delayed compared to the other streams, and the value of the delay is unknown but (supposed) constant.

This is an example of what we get : 

```
LSL-time = xdf_mouse_marker_time  :       21786.679s to       22332.342s,     48 samples for a duration of  545.662s
LSL-time = xdf_mouse_mocap_time   :       21806.683s to       22310.975s,     12 samples for a duration of  504.292s
BUG-time = xdf_kinect_marker_time :         170.238s to         839.806s,     11 samples for a duration of  669.569s
BUG-time = xdf_kinect_mocap_time  :         246.284s to         839.798s,  15667 samples for a duration of  593.514s
CSV-time = csv_kinect_mocap_time  :  1616421533.460s to  1616422126.974s,  15667 samples for a duration of  593.514s
``` 

*NOTE: the BUG has been corrected on 31 July 2023, for LSL_Kinect versions >= 1.2.0.*

# The solution
Fortunately, the kinect streams are also saved in .csv files, and the mouse streams are saved in other .csv files.

For each device, we have two streams: 
- the mocap stream, also saved in a .csv file 
- the marker stream, also saved in a .csv file 

Hence, we have :

- in the .xdf file : 
    - the kinect streams: with buggy timestamps (i.e., relative to the computer's startup time, which is unknown)
    - the mouse streams: with LSL timestamps (i.e., in sync with the other streams)

- in the .csv files :
    - the mouse streams, with timestamps as current time in milliseconds (e.g., 1616422003021 a [java currentTimeMillis](https://docs.oracle.com/javase/8/docs/api/java/lang/System.html#currentTimeMillis--))
    - the kinect streams, with timestamps as current time in milliseconds (e.g., '2021-03-22 15:06:43.021' a date-time string, corresponding to 1616422003021)

Stated differently, we have 3 time references:
- `LSL-kinect-time`: the timestamps of the Kinect streams (bugged)
- `LSL-time`: the LSL timestamps (in sync with the other streams)
- `CSV-time`: the computer's current time in JAVA milliseconds (the time reference of all .csv files)

To know the relations between the 3 time references, one solution is to compute:
- **kinect-to-csv delay**: the delay between the kinect streams and the corresponding csv files
- **mouse-to-csv delay**: the delay between mouse streams and the corresponding csv files
- **kinect-to-mouse delay**: the delay between the kinect streams and the mouse streams

From the previous, we can compute the corrected kinect timestamps : the timestamps of the kinect stream shifted by the kinect-to-mouse delay


NOTE: if the kinect and mouse are registered on the same computer, the current time is the same for both csv files. If this is not the case, we would need to take into account the time difference between the two computers clocks (which is unknown, and is the reason we use LSL...). The good news is that ReArm registrations (normally) use the same computer for running LSL-mouse and LSL-kinect. 



# The action plan, step by step

## Computing the mouse-to-csv delay
The mouse-to-csv delay is the delay between the mouse streams and the mouse csv files.

As only the mouse markers csv files are systematically saved in the ReArm data set, we  will compare the timestamps from:  
- `LSL-time` in the mouse marker stream
- `CSV-time` in the mouse markers csv file

Do do so, we need to:
- read the the `LSL-time` from the mouse marker stream 
- select the corresponding csv file (if it exists)
- read the csv file to extract the `CSV-time` column
- check that we have the same number of rows in the csv file and in the LSL stream (it should be the case)
- compare timestamps, after converting `CSV-time` to seconds
- get the mean and std of the delay

## Computing the kinect-to-csv delay
The kinect-to-csv delay is the delay between the kinect streams and the kinect csv files.

Here, we take advantage of some very good news: the first column of the kinect mocap stream contains the CSV-time. Therefore, we can simply compare the timestamps of :  
- `LSL-kinect-time` in the kinect mocap stream 
- `CSV-time` in the first column of the kinect mocap stream 

Do do so, we merely need to:
- read the the `LSL-kinect-time` from the kinect mocap stream 
- read the first column of the kinect mocap stream to extract the `CSV-time` column
- compare timestamps, after converting `CSV-time` to seconds
- get the mean and std of the delay


## Computing the kinect-to-mouse delay
The kinect-to-mouse delay is the delay between the kinect streams and the mouse streams.

To do so, we simply go from the kinect -> to csv -> to mouse time references:  
- kinect_to_mouse_delay_s = kinect_to_csv_delay_s - mouse_to_csv_delay_s

## Correcting the kinect timestamps
The corrected kinect timestamps are the timestamps of the kinect stream shifted by the kinect-to-mouse delay: 
- corrected_kinect_timestamp = LSL-kinect-time + kinect_to_mouse_delay_s



# The code for the solution

## Imports

In [ ]:
import pyxdf
import numpy as np
import tempfile
import os

# for the tests
import matplotlib.pyplot as plt

## Global initialization

In [ ]:
# this should be the set to false for production use
# doRunTests = False
# do_debug = False

# and set to true for testing (default) but can be already set to false from outside (e.g., by the test script)
if "doRunTests" not in globals():
    doRunTests = True
    do_debug = True

if doRunTests:
    # for plots with a zoom button (within VSCode)
    %matplotlib widget 

    # the best for debug-test plots (external window that you can make fullscreen)
    # %matplotlib qt 
    
    ##########################################################################################
    xdf_full_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1"
    xdf_full_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210715_V3"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210716_V1"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20210820_V2"
    # visit_path = "../dat/ReArm.lnk/ReArm_C1P07/ReArm_C1P07_20211116_V3"
    ##########################################################################################

## Kinect-to-csv delay

The kinect-to-csv delay is easy to compute: it is the time difference between the `LSL-kinect-time` and the `TimeSpan` column in the kinect mocap stream.

### Read the mouse and kinect streams from the .xdf file

NOTE: For the data recorded with the event IDE software (after patient 23 os so), the mouse streams are not present for the reaching task.

In [ ]:
def read_xdf_mouse_kinect(xdf_fullFname):
    """Read the XDF file and return a dict containing :

    - "xdf_mouse_marker_time": np.array 1D
    - "xdf_mouse_marker_data": list of makers [str]
    - "xdf_NIC_Quality_time": np.array 1D
    - "xdf_mouse_mocap_time": np.array 1D
    - "xdf_kinect_marker_time": np.array 1D
    - "xdf_kinect_mocap_time": np.array 1D
    - "csv_kinect_mocap_time": np.array 1D

    """

    if 1 == 2:
        # explore what is in the XDF file
        xdf_data, header = pyxdf.load_xdf(
            filename=xdf_fullFname,
            synchronize_clocks=True,
            dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
        )

        # print the stream names and types
        for stream in xdf_data:
            print(stream["info"]["name"][0], stream["info"]["type"][0])

    # NOTE: do not forget to synchronize the clocks for all streams
    xdf_data, header = pyxdf.load_xdf(
        filename=xdf_fullFname,
        select_streams=[
            {"type": "MoCap", "name": "EuroMov-Mocap-Kinect"},
            {"type": "MoCap", "name": "Mouse"},
            {"type": "MoCap", "name": "MouseData"},  # for the new files
            {"type": "Markers", "name": "EuroMov-Markers-Kinect"},
            {"type": "Markers", "name": "Mouse"},
            {"type": "Markers", "name": "MouseMarkers"},
            {"type": "Quality", "name": "NIC-Quality"},
        ],
        synchronize_clocks=True,
        dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
    )

    # print the stream names and types
    print("Selected streams:")
    for stream in xdf_data:
        print("  ", stream["info"]["name"][0], stream["info"]["type"][0])

    # get the streams
    kinect_mocap = [
        stream
        for stream in xdf_data
        if stream["info"]["name"][0] == "EuroMov-Mocap-Kinect"
    ]

    kinect_markers = [
        stream
        for stream in xdf_data
        if stream["info"]["name"][0] == "EuroMov-Markers-Kinect"
    ]

    mouse_markers = [
        xdf_stream
        for xdf_stream in xdf_data
        if "Mouse" in xdf_stream["info"]["name"][0]
        and xdf_stream["info"]["type"][0] == "Markers"
    ]

    mouse_mocap = [
        xdf_stream
        for xdf_stream in xdf_data
        if xdf_stream["info"]["name"][0] == "Mouse"
        or xdf_stream["info"]["name"][0] == "MouseData"
        and xdf_stream["info"]["type"][0] == "MoCap"
    ]

    NIC_Quality = [
        xdf_stream
        for xdf_stream in xdf_data
        if xdf_stream["info"]["name"][0] == "NIC-Quality"
        and xdf_stream["info"]["type"][0] == "Quality"
    ]

    # Initialize the return values
    xdf_mouse_marker_time = np.array([-1.0])
    xdf_mouse_marker_data = []
    xdf_mouse_mocap_time = np.array([-1.0])
    xdf_kinect_marker_time = np.array([-1.0])
    xdf_kinect_mocap_time = np.array([-1.0])
    csv_kinect_mocap_time = np.array([-1.0])
    xdf_NIC_Quality_time = np.array([-1.0])

    def get_time_stamps(stream):
        if stream:
            return stream[0]["time_stamps"]
        return np.array([-1.0])

    if kinect_mocap:
        kinect_mocap = kinect_mocap[0]
        xdf_kinect_mocap_time = kinect_mocap["time_stamps"]
        csv_kinect_mocap_time = kinect_mocap["time_series"][:, 0] / 1000.0

    if kinect_markers:
        kinect_markers = kinect_markers[0]
        xdf_kinect_marker_time = kinect_markers["time_stamps"]

    if mouse_markers:
        mouse_markers = mouse_markers[0]
        xdf_mouse_marker_time = mouse_markers["time_stamps"]
        xdf_mouse_marker_data = mouse_markers["time_series"]

    if mouse_mocap:
        mouse_mocap = mouse_mocap[0]
        xdf_mouse_mocap_time = mouse_mocap["time_stamps"]

    if NIC_Quality:
        NIC_Quality = NIC_Quality[0]
        xdf_NIC_Quality_time = NIC_Quality["time_stamps"]

    return {
        "xdf_mouse_marker_time": xdf_mouse_marker_time,
        "xdf_mouse_marker_data": xdf_mouse_marker_data,
        "xdf_mouse_mocap_time": xdf_mouse_mocap_time,
        "xdf_kinect_marker_time": xdf_kinect_marker_time,
        "xdf_kinect_mocap_time": xdf_kinect_mocap_time,
        "csv_kinect_mocap_time": csv_kinect_mocap_time,
        "xdf_NIC_Quality_time": xdf_NIC_Quality_time,
    }


if doRunTests:

    def test_read_xdf_mouse_kinect():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Reaching/ReArm_C1P42_20240603_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"

        xdf_mouse_kinect = read_xdf_mouse_kinect(xdf_fullFname)
        xdf_mouse_marker_time = xdf_mouse_kinect["xdf_mouse_marker_time"]
        xdf_mouse_mocap_time = xdf_mouse_kinect["xdf_mouse_mocap_time"]
        xdf_NIC_Quality_time = xdf_mouse_kinect["xdf_NIC_Quality_time"]
        xdf_kinect_marker_time = xdf_mouse_kinect["xdf_kinect_marker_time"]
        xdf_kinect_mocap_time = xdf_mouse_kinect["xdf_kinect_mocap_time"]
        csv_kinect_mocap_time = xdf_mouse_kinect["csv_kinect_mocap_time"]

        def print_stats(name, data):
            print(
                f"{name}: {data[0]:15.3f}s to {data[-1]:15.3f}s, {len(data):6.0f} samples for a duration of {data[-1] - data[0]:8.3f}s"
            )

        print("----")
        print_stats("LSL-time = xdf_mouse_marker_time  ", xdf_mouse_marker_time)
        print_stats("LSL-time = xdf_mouse_mocap_time   ", xdf_mouse_mocap_time)
        print_stats("LSL-time = xdf_NIC_Quality_time   ", xdf_NIC_Quality_time)
        print_stats("BUG-time = xdf_kinect_mocap_time  ", xdf_kinect_mocap_time)
        print_stats("BUG-time = xdf_kinect_marker_time ", xdf_kinect_marker_time)
        print_stats("CSV-time = csv_kinect_mocap_time  ", csv_kinect_mocap_time)
        print("NOTE: Negative time values indicate missing streams")

    test_read_xdf_mouse_kinect()

### Get the kinect-to-csv delay


In [ ]:
def get_kinect_to_csv_delay(xdf_fullFname):
    """get the time difference between the xdf and csv mocap time"""

    xdf_mouse_kinect = read_xdf_mouse_kinect(xdf_fullFname)
    xdf_kinect_mocap_time = xdf_mouse_kinect["xdf_kinect_mocap_time"]
    csv_kinect_mocap_time = xdf_mouse_kinect["csv_kinect_mocap_time"]

    kinect_to_csv_delay = csv_kinect_mocap_time - xdf_kinect_mocap_time
    return kinect_to_csv_delay


if doRunTests:

    def test_get_kinect_to_csv_delay():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"

        kinect_to_csv_delay = get_kinect_to_csv_delay(xdf_fullFname)

        print(f"kinect_to_csv_delay (mean): {np.mean(kinect_to_csv_delay):.3f} s")
        print(f"kinect_to_csv_delay  (std): {np.std(kinect_to_csv_delay):.6f} s")

        # plot the time difference
        plt.figure()
        t = np.arange(len(kinect_to_csv_delay))
        x = kinect_to_csv_delay - np.mean(kinect_to_csv_delay)
        plt.plot(t, x, ".", markersize=1)
        plt.title("Kinect to CSV time difference (mean centered)")
        plt.xlabel("Frame number")
        plt.ylabel("Time difference (s)")
        plt.grid()
        plt.show()

    test_get_kinect_to_csv_delay()

## Mouse to csv delay

The task is not as trivial as computing the kinect-to-csv delay, as we need to find the corresponding csv file(s) for the mouse marker stream.


### Read one markers csv file

In [ ]:
def read_marker_csv_file(full_fname_marker_csv):
    """Read one marker csv file and return a list of [timestamp, marker] pairs

    Parameters
    ----------
    full_fname_marker_csv : str
        Full filename of the marker csv file

    Returns
    -------

    timestamp_marker_list : list
        List of [timestamp, marker] pairs
            timestamp : float (in seconds)
            marker : str
    """

    if not full_fname_marker_csv.endswith(".csv"):
        raise ValueError("The file must be a csv file")

    # NOTE: some mouse markers lack the quotes around the multiline markers
    # we need to add the quotes around the multiline markers before loading the file
    with open(full_fname_marker_csv, "r") as fname:
        txt = fname.readlines()

    lines = [line.split(",") for line in txt]

    # find the lines where token 3 is "\n" = start of a multiline marker
    for i in range(len(lines)):
        line = lines[i]
        # find the start of a multiline marker
        if len(line) == 3 and line[2] == "\n":
            # add a " before the end of the line
            txt[i] = txt[i][:-1] + '"\n'
            # find the end of the multiline marker
            for j in range(i + 1, len(lines)):
                # if we have a normal one-line-marker
                if len(lines[j]) == 3:
                    # add a " before the end of the line (of the previous line)
                    txt[j - 1] = txt[j - 1][:-1] + '"\n'
                    break
                # if are at the end of the file
                if j == len(lines) - 1 and len(lines[j]) != 3:
                    # add a " before the end of the line
                    txt[j] = txt[j][:-1] + '"\n'
                    break

    # write the modified file to a temporary file and load it with np.loadtxt
    with tempfile.NamedTemporaryFile(mode="w", delete=False) as fname:
        fname.writelines(txt)
        tempFileName = fname.name

    lines = np.loadtxt(
        fname=tempFileName, skiprows=3, delimiter=",", quotechar='"', dtype=str
    )

    # remove the first column (we shall need only the timestamp and the marker to compare with the xdf file)
    lines = np.delete(lines, 0, 1)

    # return a list of [timestamp, marker] pairs
    timestamp_marker_list = []
    for line in lines:
        # WARNING : the timestamp must be in seconds (as in xdf files)
        timestamp = float(line[0]) / 1000
        marker = line[1]
        timestamp_marker_list.append([timestamp, [marker]])

    return timestamp_marker_list


def plot_markers_csv_time_difference(marker_list, title_txt=""):
    """plot the time difference between the markers in the list"""
    plt.figure()
    t = np.arange(len(marker_list))
    x = [data[0] for data in marker_list]
    dx = np.diff(x)
    dx = np.insert(dx, 0, np.nan)

    plt.plot(t, dx, ".", markersize=10)
    plt.xlabel("Marker")
    plt.ylabel("Time difference (s)")
    plt.grid()
    # set the xticks to the marker names
    xticks_labels = [data[1][0] for data in marker_list]
    plt.xticks(t, xticks_labels, rotation=90, ha="center", fontsize=8)
    # leave some room in the bottom for the xticks_labels
    plt.subplots_adjust(bottom=0.5)
    # set the title
    plt_title = "full-screen window to view Markers labels"
    if title_txt:
        plt_title = f"{title_txt}\n{plt_title}"
    plt.title(plt_title)

    plt.show()

    print(f"{title_txt}: time difference {len(marker_list)} markers")
    print("  DeltaT     Time        Marker")
    for i in range(len(marker_list)):
        print(f"{dx[i]:8.3f} {marker_list[i][0]:.3f} {marker_list[i][1]}")


if doRunTests:

    def test_readMarkerCsv():
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_mau_np.csv"
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_mau_p.csv"
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_sau_np.csv"
        csv_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r_l_m_sau_p.csv"

        fpath, fname = os.path.split(csv_fullFname)
        title_txt = f"{fname}"

        csv_marker_data = read_marker_csv_file(csv_fullFname)
        plot_markers_csv_time_difference(csv_marker_data, title_txt)

    test_readMarkerCsv()

### Read all markers csv files in the folder

In [ ]:
def read_all_marker_csv_files(xdf_full_path):
    """Read all marker csv files in the visit path and return a list of [timestamp, marker]
    pairs

    Parameters
    ----------
    xdf_full_path : str
        Full path of the directory where the xdf file is located

    Returns
    -------
    all_timestamp_marker_list : list
        List of [timestamp, marker] pairs
            timestamp : float (in seconds)
            marker : str
    """

    # get the list of all marker csv files in the visit path
    marker_files = [
        os.path.join(xdf_full_path, f)
        for f in os.listdir(xdf_full_path)
        if f.endswith(".csv") and "_l_m_" in f
    ]

    # read all the marker csv files
    marker_data_dict = {}
    for i in range(len(marker_files)):
        mouse_markers = read_marker_csv_file(marker_files[i])
        marker = {
            "data": mouse_markers,
            "start": mouse_markers[0][0],
            "path": marker_files[i],
        }
        marker_data_dict[i] = marker

    # sort the markers by their start time
    sorted_marker_data = sorted(marker_data_dict.items(), key=lambda x: x[1]["start"])

    # make a single list of markers from the multiple csv files for this xdf file
    all_timestamp_marker_list = []
    for i in range(len(sorted_marker_data)):
        all_timestamp_marker_list.extend(sorted_marker_data[i][1]["data"])

    return all_timestamp_marker_list


if doRunTests:

    def test_readAllMarkerCsvs():
        xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/"
        xdf_path = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle"

        xdf_path = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular"

        xdf_dir = xdf_path.split("/")[-1]  # get the last part of the path

        all_marker_data = read_all_marker_csv_files(xdf_path)
        plot_markers_csv_time_difference(
            all_marker_data, f"{xdf_dir}/ — Mouse Markers from all csv files"
        )

    test_readAllMarkerCsvs()

### Read the mouse and kinect streams from one .xdf file

In [ ]:
def read_xdf_mouse_markers(xdf_full_fname):
    """Read the mouse marker stream in an xdf file and return a list of [timestamp, marker]
    pairs

    Parameters
    ----------
    xdf_full_fname : str
        Full filename of the xdf file

    Returns
    -------
    timestamp_marker_list : list
        List of [timestamp, marker] pairs
            timestamp : float (in seconds)
            marker : str
    """

    xdf_data, header = pyxdf.load_xdf(
        filename=xdf_full_fname,
        select_streams=[
            {"type": "Markers", "name": "Mouse"},
            {"type": "Markers", "name": "MouseMarkers"},  # new naming possible
        ],
        synchronize_clocks=True,
        dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
    )

    mouse_markers = [
        xdf_stream
        for xdf_stream in xdf_data
        if xdf_stream["info"]["name"][0] == "Mouse"
        or xdf_stream["info"]["name"][0] == "MouseMarkers"
    ][0]

    xdf_mouse_marker_time = mouse_markers["time_stamps"]
    xdf_mouse_marker_data = mouse_markers["time_series"]

    # return a list of [timestamp, marker] pairs
    timestamp_marker_list = []
    for i in range(len(xdf_mouse_marker_time)):
        timestamp_marker_list.append(
            [xdf_mouse_marker_time[i], xdf_mouse_marker_data[i]]
        )

    return timestamp_marker_list


if doRunTests:

    def test_xdf_mouse_markers():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"

        fname = xdf_fullFname.split("/")[-1]
        title_txt = f"{fname} — Mouse Markers"

        xdf_mouse_maker_list = read_xdf_mouse_markers(xdf_fullFname)
        plot_markers_csv_time_difference(xdf_mouse_maker_list, title_txt)

    test_xdf_mouse_markers()

### [Functions to compare [timestamp, marker ] lists](#toc0_)

The human-readable markers streams are stored in lists of [timestamp, marker].   
We need to compare the list from the xdf and the list from the csv file(s) to ensure that we map the correct markers to the correct timestamps in both lists.


In [ ]:
def define_shortest_longest_lists(list1, list2):
    """Define the shortest and the longest list"""

    shortest_list = list1
    longest_list = list2
    if len(list2) < len(list1):
        shortest_list = list2
        longest_list = list1

    return shortest_list, longest_list


def find_first_occurrence_of_short_list_in_long_list(shortest_list, longest_list):
    """Find the first occurrence of the shortest list in the longest list
    the lists are lists of markers, hence comparison is on list[i][1] (the marker)"""

    if do_debug:

        # create a debug directory if it does not exist
        if not os.path.exists("../debug"):
            os.makedirs("../debug")

        # save the shortest list to a file for debugging
        with open("../debug/shortest_list.txt", "w") as f:
            for item in shortest_list:
                marker = item[1]
                f.write(f"{marker}\n")

        # save the longest list to a file for debugging
        with open("../debug/longest_list.txt", "w") as f:
            for item in longest_list:
                marker = item[1]
                f.write(f"{marker}\n")

    i_beg = -1
    i_end = -1
    error_msg = ""
    for i in range(len(longest_list)):
        longest_list_marker = longest_list[i][1]
        shortest_list_marker = shortest_list[0][1]
        start_found = (longest_list_marker == shortest_list_marker) and i_beg == -1
        if start_found:
            i_beg = i
            nb_j = len(shortest_list)
            for j in range(len(shortest_list)):
                if longest_list[i + j][1] == shortest_list[j][1]:
                    i_end = i + j
                # if we are at the end of the list, we have found the end
                if j == len(shortest_list) - 1:
                    break

    if i_beg == -1:
        error_msg += "The shortest list is not found in the longest list\n"
    if i_end == -1:
        error_msg += "The end of the shortest list is not found in the longest list\n"

    return i_beg, i_end, error_msg


def get_timestamps_differences(longest_list, shortest_list, i_beg, i_end):
    """Get the differences between the timestamps of the common part and the shortest list"""

    if len(shortest_list) > len(longest_list):
        raise ValueError(
            "The shortest list (second argument) must be shorter than the longest list"
        )

    timestamps_common_part = [x[0] for x in longest_list[i_beg : i_end + 1]]
    timestamps_shortest_list = [x[0] for x in shortest_list]

    timestamps_differences = [
        timestamps_common_part[i] - timestamps_shortest_list[i]
        for i in range(len(shortest_list))
    ]

    # NOTE: we do not know whether csv or xdf is in the shortest list, hence we do not know the sign!
    # Too bad... BUT...
    # timestamps_differences MUST be positive values.
    # This is because csv time is UNIX time (seconds since 1970) and
    # xdf time is in seconds since the start of the recording (or something like that).

    timestamps_differences = [abs(x) for x in timestamps_differences]

    return timestamps_differences

### Get the mouse-to-csv delay

In [ ]:
def get_mouse_to_csv_delay_list(xdf_mouse_marker_list, csv_mouse_marker_list):
    """Get the mouse-to-csv delay as a list of [timestamp, marker] pairs"""

    # find the shortest and the longest list
    shortest_list, longest_list = define_shortest_longest_lists(
        xdf_mouse_marker_list, csv_mouse_marker_list
    )

    # find the first occurrence of the shortest list in the longest list
    i_beg, i_end, error_msg = find_first_occurrence_of_short_list_in_long_list(
        shortest_list, longest_list
    )

    if i_beg == -1 or i_end == -1:
        raise ValueError(error_msg)

    # get the differences between the timestamps of the common part and the shortest list
    timestamps_differences = get_timestamps_differences(
        longest_list, shortest_list, i_beg, i_end
    )
    # as np array
    timestamps_differences = np.array(timestamps_differences)

    # get the makers corresponding to the timestamps_differences
    markers_differences = [longest_list[i][1] for i in range(i_beg, i_end + 1)]

    # make it a list of [timestamp, marker] pairs
    timestamps_differences = list(zip(timestamps_differences, markers_differences))

    return timestamps_differences


if doRunTests:

    def test_get_mouse_to_csv_delay():

        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"

        xdf_path = os.path.dirname(xdf_fullFname)

        # get the csv marker data
        csv_mouse_marker_list = read_all_marker_csv_files(xdf_path)
        csv_mouse_marker_time = [data[0] for data in csv_mouse_marker_list]
        csv_mouse_marker_data = [data[1] for data in csv_mouse_marker_list]

        xdf_mouse_kinect = read_xdf_mouse_kinect(xdf_fullFname)
        xdf_mouse_marker_time = xdf_mouse_kinect["xdf_mouse_marker_time"]
        xdf_mouse_marker_data = xdf_mouse_kinect["xdf_mouse_marker_data"]

        xdf_mouse_marker_list = list(zip(xdf_mouse_marker_time, xdf_mouse_marker_data))

        # get the mouse-to-csv delay as a list of [timestamp, marker] pairs
        mouse_to_csv_delay_list = get_mouse_to_csv_delay_list(
            xdf_mouse_marker_list, csv_mouse_marker_list
        )

        # get the mouse-to-csv delay timestamps
        mouse_to_csv_delay = [data[0] for data in mouse_to_csv_delay_list]
        # as a np array
        mouse_to_csv_delay = np.array(mouse_to_csv_delay)

        print(f"mouse_to_csv_delay (median): {np.median(mouse_to_csv_delay):.3f} s")
        print(f"mouse_to_csv_delay   (mean): {np.mean(mouse_to_csv_delay):.3f} s")
        print(f"mouse_to_csv_delay    (std): {np.std(mouse_to_csv_delay):.6f} s")

        # plot the time difference
        t = np.arange(len(mouse_to_csv_delay))
        x = mouse_to_csv_delay - np.mean(mouse_to_csv_delay)

        fig, ax = plt.subplots(1, 2, width_ratios=[10, 1], figsize=(12, 6))
        # scatter plot
        ax[0].plot(t, x, ".", markersize=10)
        ax[0].set_title(
            f"Mouse to CSV time difference mean = {np.mean(mouse_to_csv_delay):.3f} s"
        )
        ax[0].set_xlabel("Marker number")
        ax[0].set_ylabel("Time difference (mean centered, s)")
        ax[0].grid()

        # leave some room in the bottom for the xticks_labels
        plt.subplots_adjust(bottom=0.5)
        # add the xticks_labels
        xticks_labels = [data[1][0] for data in mouse_to_csv_delay_list]
        ax[0].set_xticks(t)
        ax[0].set_xticklabels(xticks_labels, rotation=90, ha="center", fontsize=8)

        # boxplot
        ax[1].boxplot(x, showmeans=True)
        # # remove the frame around the boxplot
        ax[1].axis("off")
        # ensure the x limits are the same for both plots
        ax[1].set_ylim(ax[0].get_ylim())

        plt.show()

    test_get_mouse_to_csv_delay()

## Kinect-to-mouse delay

In [ ]:
# get the kinect-to-mouse delay
#   kinect_to_mouse_delay_s = kinect_to_csv_delay_s - mouse_to_csv_delay_s


def get_kinect_to_mouse_delay(kinect_to_csv_delay, mouse_to_csv_delay):
    """Get the kinect-to-mouse delay"""

    kinect_to_cvs_delay_median = np.median(kinect_to_csv_delay)
    kinect_to_cvs_delay_mean = np.mean(kinect_to_csv_delay)
    kinect_to_cvs_delay_std = np.std(kinect_to_csv_delay)

    mouse_to_csv_delay_median = np.median(mouse_to_csv_delay)
    mouse_to_csv_delay_mean = np.mean(mouse_to_csv_delay)
    mouse_to_csv_delay_std = np.std(mouse_to_csv_delay)

    # NOTE: the kinect to mouse delay is the difference of the means-medians
    # The mean & std is the simplest approach, if the distribution is normal
    # which should be true, as the error is due to random delays in the system
    kinect_to_mouse_delay_mean = kinect_to_cvs_delay_mean - mouse_to_csv_delay_mean

    kinect_to_mouse_delay_median = (
        kinect_to_cvs_delay_median - mouse_to_csv_delay_median
    )

    # NOTE: the variance of the difference is the sum of the variances **minus the covariance**
    # (e.g. https://en.wikipedia.org/wiki/Propagation_of_uncertainty)
    # If we assume that the two distributions are independent, the covariance is zero
    # hence computing the variance of the difference as the sum of the variances is correct.
    # If we assume that the two distributions are not independent, we should subtract the covariance,
    # but we do not have it. We still can compute the variance of the difference as the sum of the variances,
    # and this will be an **upper bound of the variance of the difference**.
    kinect_to_mouse_delay_std = np.sqrt(
        kinect_to_cvs_delay_std**2 + mouse_to_csv_delay_std**2
    )

    return (
        kinect_to_mouse_delay_mean,
        kinect_to_mouse_delay_median,
        kinect_to_mouse_delay_std,
    )


def plot_delay_distribution(
    kinect_to_csv_delay,
    mouse_to_csv_delay,
):
    """Boxplot the distribution of the kinect-to-csv and mouse-to-csv delays"""

    # boxplot the two distributions (mean centered)
    k_distrib = kinect_to_csv_delay - np.mean(kinect_to_csv_delay)
    m_distrib = mouse_to_csv_delay - np.mean(mouse_to_csv_delay)

    plt.figure()
    plt.boxplot([k_distrib, m_distrib], showmeans=True)
    plt.title("Kinect to CSV delay vs Mouse to CSV delay")
    plt.xticks([1, 2], ["Kinect to CSV", "Mouse to CSV"])
    plt.ylabel("Time difference, mean centered (s)")
    plt.grid()
    # add the 95% confidence interval

    kinect_to_cvs_delay_std = np.std(kinect_to_csv_delay)
    mouse_to_csv_delay_std = np.std(mouse_to_csv_delay)

    plt.errorbar(
        [1.2, 2.2],
        [0, 0],
        yerr=[1.96 * kinect_to_cvs_delay_std, 1.96 * mouse_to_csv_delay_std],
        fmt="o",
        color="b",
        label="95% confidence interval",
    )
    plt.legend()
    plt.show()


def get_kinect_to_mouse_delay_for_xdf(xdf_fullFname):
    """Get the kinect-to-mouse delay for the xdf file in a dict containing :
    - "kinect_to_mouse_delay_mean": float
    - "kinect_to_mouse_delay_median": float
    - "kinect_to_mouse_delay_std": float
    - "kinect_to_csv_delay": np.array
    - "mouse_to_csv_delay": np.array
    """
    xdf_path = os.path.dirname(xdf_fullFname)

    # get the csv marker data
    csv_mouse_marker_list = read_all_marker_csv_files(xdf_path)

    xdf_mouse_kinect = read_xdf_mouse_kinect(xdf_fullFname)
    xdf_mouse_marker_time = xdf_mouse_kinect["xdf_mouse_marker_time"]
    xdf_mouse_marker_data = xdf_mouse_kinect["xdf_mouse_marker_data"]

    xdf_mouse_marker_list = list(zip(xdf_mouse_marker_time, xdf_mouse_marker_data))

    # get the mouse-to-csv delay
    mouse_to_csv_delay_list = get_mouse_to_csv_delay_list(
        xdf_mouse_marker_list, csv_mouse_marker_list
    )
    mouse_to_csv_delay = np.array([data[0] for data in mouse_to_csv_delay_list])

    # get the kinect-to-csv delay
    kinect_to_csv_delay = get_kinect_to_csv_delay(xdf_fullFname)

    # get the kinect-to-mouse delay
    (
        kinect_to_mouse_delay_mean,
        kinect_to_mouse_delay_median,
        kinect_to_mouse_delay_std,
    ) = get_kinect_to_mouse_delay(kinect_to_csv_delay, mouse_to_csv_delay)

    return {
        "kinect_to_mouse_delay_mean": kinect_to_mouse_delay_mean,
        "kinect_to_mouse_delay_median": kinect_to_mouse_delay_median,
        "kinect_to_mouse_delay_std": kinect_to_mouse_delay_std,
        "kinect_to_csv_delay": kinect_to_csv_delay,
        "mouse_to_csv_delay": mouse_to_csv_delay,
    }


if doRunTests:

    def test_get_kinect_to_mouse_time_delay():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2/ReArm_C1P02_20210419_V2_Circle/ReArm_C1P02_20210409_V2_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2/ReArm_C1P02_20210419_V2_Reaching/ReArm_C1P02_20210409_V2_r.xdf"

        delays_for_xdf = get_kinect_to_mouse_delay_for_xdf(xdf_fullFname)
        kinect_to_mouse_delay_mean = delays_for_xdf["kinect_to_mouse_delay_mean"]
        kinect_to_mouse_delay_median = delays_for_xdf["kinect_to_mouse_delay_median"]
        kinect_to_mouse_delay_std = delays_for_xdf["kinect_to_mouse_delay_std"]
        kinect_to_csv_delay = delays_for_xdf["kinect_to_csv_delay"]
        mouse_to_csv_delay = delays_for_xdf["mouse_to_csv_delay"]

        print(f"kinect_to_mouse_delay (median): {kinect_to_mouse_delay_median:.3f} s")
        print(f"kinect_to_mouse_delay   (mean): {kinect_to_mouse_delay_mean:.3f} s")
        print(f"kinect_to_mouse_delay    (std): {kinect_to_mouse_delay_std:.6f} s")

        # display the kinect to mouse delay
        plot_delay_distribution(
            kinect_to_csv_delay,
            mouse_to_csv_delay,
        )

    test_get_kinect_to_mouse_time_delay()

# Test the solution on one file



## Utility functions for .xdf files

In [ ]:
def print_streams_names_type(xdf_fullFname):
    """Print the names and types of all streams in the xdf file"""

    xdf_data, header = pyxdf.load_xdf(filename=xdf_fullFname)

    for i in range(len(xdf_data)):
        stream = xdf_data[i]
        s_type = stream["info"]["type"][0]
        s_name = stream["info"]["name"][0]
        print(f"Stream {i}: {s_type}, {s_name}")


def get_stream(xdf_data, stream_type, stream_name):
    """Get the stream data from the xdf_data"""

    stream = [
        stream
        for stream in xdf_data
        if stream["info"]["type"][0] == stream_type
        # to allow partial name ("Mouse" "MouseMarker", "MouseData")
        and stream_name in stream["info"]["name"][0]
    ]
    if not stream:
        raise ValueError(f"Stream {stream_name} not found in the xdf data")

    return stream[0]


def get_kinect_channel(kinect_mocap, channel_name):
    """Get one channel from the kinect mocap by its name"""
    channel_index = -1
    nb_channels = len(kinect_mocap["info"]["desc"][0]["channels"][0]["channel"])
    for i in range(nb_channels):
        current_name = kinect_mocap["info"]["desc"][0]["channels"][0]["channel"][i][
            "label"
        ][0]
        if current_name == channel_name:
            channel_index = i
            break
    if channel_index == -1:
        raise ValueError(f"Joint {channel_name} not found in the kinect mocap data")

    channel_data = kinect_mocap["time_series"][:, channel_index]

    return channel_data

## Check if the xdf file has a kinect stream with bugged timestamps


 There is a bug if the range of the kinect timestamps differs from the range of the other LSL timestamps.

 We use the NIC Quality stream as a reference, as it is always present in the ReArm data set.

 Typical differences are: 

 * If there is a bug: 
    ```
    LSL-time = xdf_NIC_Quality_time   :       22886.697s to       23270.697s,    385 samples for a duration of  384.000s 
    BUG-time = xdf_kinect_mocap_time  :          39.798s to         416.469s,  11277 samples for a duration of  376.671s 
    ```

* If there is no bug: 
    ```
    LSL-time = xdf_NIC_Quality_time   :     3629041.763s to     3629294.763s,    254 samples for a duration of  253.000s
    BUG-time = xdf_kinect_mocap_time  :     3629041.174s to     3629295.670s,   5622 samples for a duration of  254.496s
    ``` 

We decide that there is no bug if the difference between the two ranges is less than 1 hour (3600s).

In [ ]:
def needs_kinect_timestamps_correction(xdf_fullFame):
    """
    Check if the timestamps of the Kinect stream need to be corrected.
    The timestamps of the Kinect stream are compared to the timestamps of the NIC-Quality stream.
    The timestamps of the NIC-Quality stream are considered as the reference.

    returns:
    - True if the timestamps of the Kinect stream need to be corrected
    - False if the timestamps of the Kinect stream are in the range of the NIC-Quality timestamps
    """

    # NOTE: we must not use the dejitter_timestamps=True because :
    #  - the EuroMov-Mocap-Kinect stream is not a regular stream (either 30Hz or 15 Hz, depending on the light conditions)
    #  - the dejitter_timestamps=True will interpolate the timestamps of the Kinect stream (we do not want that)
    #  - we want to compare the raw timestamps of the Kinect stream with the raw timestamps of the NIC-Quality stream

    streams, fileheader = pyxdf.load_xdf(
        filename=xdf_fullFame,
        select_streams=[  # select only the streams we need = fast
            {"type": "MoCap", "name": "EuroMov-Mocap-Kinect"},
            {"type": "Quality", "name": "NIC-Quality"},
        ],
        synchronize_clocks=True,
        dejitter_timestamps=False,  # to get the raw timestamps
    )

    # NIC-Quality stream is always present in the xdf files
    nic_quality_stream = [
        stream for stream in streams if stream["info"]["name"][0] == "NIC-Quality"
    ]
    kinect_stream = [
        stream
        for stream in streams
        if stream["info"]["name"][0] == "EuroMov-Mocap-Kinect"
    ]

    # get the timestamps
    nic_quality_timestamps = nic_quality_stream[0]["time_stamps"]
    kinect_timestamps = kinect_stream[0]["time_stamps"]

    # check the difference between the two timestamps
    delta_timestamps_zero = nic_quality_timestamps[0] - kinect_timestamps[0]
    delta_timestamps_end = nic_quality_timestamps[-1] - kinect_timestamps[-1]
    print("delta timestamps from EuroMov-Mocap-Kinect to NIC-Quality:")
    print(f" beg : {delta_timestamps_zero:8.3f}s ")
    print(f" end : {delta_timestamps_end:8.3f}s ")

    # checking the first and last timestamps is enough to detect a problem
    max_delta = 3600  # 1 hour
    needs_correction = (
        np.abs(delta_timestamps_zero) > max_delta
        or np.abs(delta_timestamps_end) > max_delta
    )

    return needs_correction


if doRunTests:

    def test_check_kinect_timestamps():
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Reaching/ReArm_C1P02_20210322_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"

        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Reaching/ReArm_C1P42_20240603_V1_r.xdf"
        xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"

        needs_correction = needs_kinect_timestamps_correction(xdf_fullFname)
        if needs_correction:
            print("ERROR: delta_timestamps > 3600")
        else:
            print("Kinect timestamps are in the range of NIC-Quality timestamps")

    test_check_kinect_timestamps()

## Test the solution on one .xdf circle file

The circle tasks records the hand movement of the participant drawing a circle. The same hand movement is recorded by the kinect and the mouse, hence allowing visual inspection of the corrected kinect timestamps.

In [ ]:
if doRunTests:
    from scipy.signal import butter, filtfilt

    def plot_t_xyz(t, x, y, z, name):
        """Plot the x, y, z data as a function of time", name is the name of the data"""
        plt.plot(t, x, ".", markersize=2, label=f"x.{name}")
        plt.plot(t, y, ".", markersize=2, label=f"y.{name}")
        plt.plot(t, z, ".", markersize=2, label=f"z.{name}")
        plt.xlabel("Time (s)")
        plt.ylabel("Position (m)")
        plt.legend()
        plt.grid()

    def plot_trajectory(t, x, y, z, t_start, t_stop, name):
        """Plot the 3D trajectory of the x, y, z data"""

        # restrict the zone of interest from t_start to t_stop
        i_start = np.where(t >= t_start)[0][0]
        i_stop = np.where(t <= t_stop)[0][-1]
        x = x[i_start:i_stop]
        y = y[i_start:i_stop]
        z = z[i_start:i_stop]

        fig = plt.figure()
        ax = fig.add_subplot(111, projection="3d")
        ax.plot(x, y, z, ".", markersize=2)
        ax.set_xlabel(f"x.{name}")
        ax.set_ylabel(f"y.{name}")
        ax.set_zlabel(f"z.{name}")
        plt.title(f"{name} 3D trajectory")
        plt.show()

    def low_pass_butt(signal, cutoff, fs, order=2):
        """Low pass filter the signal using a Butterworth filter"""
        nyquist = 0.5 * fs
        normal_cutoff = cutoff / nyquist
        b, a = butter(N=order, Wn=normal_cutoff, btype="low", analog=False)
        y = filtfilt(b, a, signal)
        return y

    def test_correct_xdf_kinect_mocap_time():

        # NOTE: only circle task has a real record of the mouse motion
        xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210419_V2/ReArm_C1P02_20210419_V2_Circle/ReArm_C1P02_20210409_V2_c.xdf"
        # xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"

        # NOTE: to get the exact name and type of the streams in the xdf file
        print_streams_names_type(xdf_fullFname)

        # NOTE: only read what you need <=> super fast
        xdf_data, header = pyxdf.load_xdf(
            filename=xdf_fullFname,
            synchronize_clocks=True,
            select_streams=[
                {"type": "MoCap", "name": "EuroMov-Mocap-Kinect"},
                {"type": "Markers", "name": "EuroMov-Markers-Kinect"},
                {"type": "Markers", "name": "Mouse"},
                {"type": "MoCap", "name": "Mouse"},
                {"type": "Markers", "name": "MouseMarkers"},  # alternative name
                {"type": "MoCap", "name": "MouseData"},  # alternative name
                {"type": "Accelerometer", "name": "NIC-Accelerometer"},
            ],
            dejitter_timestamps=False,  # to get the raw timestamps to compare with the CSV
        )

        mouse_mocap = get_stream(xdf_data, "MoCap", "Mouse")
        kinect_mocap = get_stream(xdf_data, "MoCap", "EuroMov-Mocap-Kinect")
        acc_mocap = get_stream(xdf_data, "Accelerometer", "NIC-Accelerometer")

        mouse_markers = get_stream(xdf_data, "Markers", "Mouse")
        kinect_markers = get_stream(xdf_data, "Markers", "EuroMov-Markers-Kinect")

        mouse_t = mouse_mocap["time_stamps"]
        mouse_x = mouse_mocap["time_series"][:, 0]
        mouse_y = mouse_mocap["time_series"][:, 1]
        mouse_z = mouse_mocap["time_series"][:, 2]  # not used

        acc_t = acc_mocap["time_stamps"]
        acc_x = acc_mocap["time_series"][:, 0]
        acc_y = acc_mocap["time_series"][:, 1]
        acc_z = acc_mocap["time_series"][:, 2]

        kinect_t = kinect_mocap["time_stamps"]
        WristRight_X = get_kinect_channel(kinect_mocap, "WristRight_X")
        WristRight_Y = get_kinect_channel(kinect_mocap, "WristRight_Y")
        WristRight_Z = get_kinect_channel(kinect_mocap, "WristRight_Z")

        WristLeft_X = get_kinect_channel(kinect_mocap, "WristLeft_X")
        WristLeft_Y = get_kinect_channel(kinect_mocap, "WristLeft_Y")
        WristLeft_Z = get_kinect_channel(kinect_mocap, "WristLeft_Z")

        # low pass the Wrist timeseries
        fs = 30
        cutoff = 2

        WristRight_X = low_pass_butt(WristRight_X, cutoff, fs)
        WristRight_Y = low_pass_butt(WristRight_Y, cutoff, fs)
        WristRight_Z = low_pass_butt(WristRight_Z, cutoff, fs)

        WristLeft_Xf = low_pass_butt(WristLeft_X, cutoff, fs)
        WristLeft_Yf = low_pass_butt(WristLeft_Y, cutoff, fs)
        WristLeft_Zf = low_pass_butt(WristLeft_Z, cutoff, fs)

        # # plot the effect of the filter
        # plt.figure()
        # plot_t_xyz(kinect_t, WristRight_Xf, WristRight_X, WristRight_Xf, "Filtered WristRight_X")
        # plt.show()
        # # plot the effect of the filter
        # plt.figure()
        # plot_t_xyz(kinect_t, WristLeft_Xf, WristLeft_X, WristLeft_Xf, "Filtered WristLeft_X")
        # plt.show()

        # # plot the trajectory of the hand
        # plt.figure()
        # plot_trajectory(kinect_t, WristRight_Xf, WristRight_X, WristRight_Xf, 270, 295, "WristRight_270_295")
        # plt.show()

        # check if the file needs a kinect timestamp correction
        if needs_kinect_timestamps_correction(xdf_fullFname):
            delays_for_xdf = get_kinect_to_mouse_delay_for_xdf(xdf_fullFname)
            kinect_to_mouse_delay_mean = delays_for_xdf["kinect_to_mouse_delay_mean"]

            ############################################################
            # correct the xdf_kinect_mocap_time
            kinect_t_correct = kinect_t + kinect_to_mouse_delay_mean
            ############################################################
        else:
            kinect_t_correct = kinect_t

        # plot the mouse
        plt.figure()
        plot_t_xyz(mouse_t, mouse_x, mouse_y, mouse_z, "Mouse")
        plt.show()

        # plot the accelerometer
        plt.figure()
        plot_t_xyz(acc_t, acc_x, acc_y, acc_z, "Accelerometer")
        plt.show()

        # plot the kinect
        plt.figure()
        plot_t_xyz(kinect_t, WristRight_X, WristRight_Y, WristRight_Z, "WristRight")
        plot_t_xyz(kinect_t, WristLeft_X, WristLeft_Y, WristLeft_Z, "WristLeft")
        plt.show()

        # plot the kinect x and y motion of the hand **on the same plot***
        plt.figure()
        plot_t_xyz(
            kinect_t_correct,
            WristRight_X,
            WristRight_Y,
            WristRight_Z,
            "WristRight_correct",
        )
        scale = 0.001  # define a compatible y scale for the mouse motion
        plot_t_xyz(mouse_t, mouse_x * scale, mouse_y * scale, mouse_z * scale, "Mouse")
        plt.show()

    test_correct_xdf_kinect_mocap_time()

In [ ]:
xdf_fullFname = "../dat/ReArm.lnk/ReArm_C1P02/ReArm_C1P02_20210306_V1/ReArm_C1P02_20210306_V1_Circle/ReArm_C1P02_20210322_V1_c.xdf"
xdf_fullFname = "../dat/ReArm.lnk/Fichiers de Karima Bakhti - C1P42/V1/Circular/ReArm_C1P42_20240603_V1_c.xdf"

delays = get_kinect_to_mouse_delay_for_xdf(xdf_fullFname)
for key, value in delays.items():
    print(f"{key}: {value}")

# TODO list...

## Get the list of the needed csv files for the Reaching and Circle task

We need the list of the .csv files corresponding to each .xdf file.  
We use the list given by `goodFiles.log` in the visit directory.  
Only the Reaching and Circle tasks contain an xdf file with a kinect stream.  

NOTE : for the mouse, we shall not use the data csv files, but only the marker csv files.

The csv files that are expected are: 

- the kinect csv files (NONE are mandatory) 
    - `*_k.csv`: kinect data file 
    - `*_k_m.csv`: the kinect marker file

- the mouse csv marker files for the Reaching task (at least ONE is mandatory) 
    - `*_r_l_m_mau_np.csv`: mouse marker file for maximal arm use with the non-paretic arm
    - `*_r_l_m_mau_p.csv`: mouse marker file for maximal arm use with the paretic arm
    - `*_r_l_m_sau_np.csv`: mouse marker file for spontaneous arm use with the non-paretic arm
    - `*_r_l_m_sau_p.csv`: mouse marker file for spontaneous arm use with the paretic arm

- the mouse csv marker files for the Circle task (at least ONE is mandatory) 
    - `*_c_l_m_np.csv`: mouse marker file for the non-paretic arm
    - `*_c_l_m_p.csv`: mouse marker file for the paretic arm


The minimal set of files that are mandatory **for each task** are:
- the xdf file (the file that contains the kinect stream with wrong timestamps + the mouse stream with correct timestamps) 
- one mouse marker csv file (the file that contains the mouse stream with correct timestamps)


In [ ]:
# TODO: test the case where one or more csv files are missing

## Compute the kinect-to-mouse delay for all xdf files in the visit directory

